# core

> Shared exceptions, raster I/O, focal-neighbourhood statistics, and
> standardisation for `morphon_nbdev` -- a self-implemented (no
> WhiteboxTools/RichDEM/GRASS/SAGA) port of the Dove/Nanson two-part seabed
> geomorphology classification scheme, built for Bass Strait but usable on any
> projected bathymetry raster.

In [ ]:
#| default_exp core

In [ ]:
#| hide
#| export
from nbdev.showdoc import *

import itertools

import rioxarray  # noqa: F401 -- registers the .rio accessor on xr.DataArray
import xarray as xr
import numpy as np
import geopandas as gpd
import holoviews as hv
import hvplot.pandas  # noqa: F401 -- registers the .hvplot accessor on GeoDataFrame
import hvplot.xarray  # noqa: F401 -- registers the .hvplot accessor on xr.DataArray

from plum import dispatch
from scipy.signal import fftconvolve
from rasterio.features import shapes as rio_shapes
from shapely.geometry import shape as shapely_shape

from morphon_nbdev.exceptions import *

hv.extension("bokeh")

### `load_bathymetry`

Entry point for every other function in this module -- loads a bathymetry GeoTIFF
(or any rasterio-readable raster) into an `xr.DataArray`, converts nodata to NaN,
and enforces a projected CRS up front so every downstream focal/kernel operation
can assume real distance units (metres) rather than degrees.

In [ ]:
#| export
def load_bathymetry(path: str, band: int = 1) -> xr.DataArray:
    """Load a bathymetry raster as a 2D DataArray, nodata converted to NaN, and require a projected CRS."""
    da = rioxarray.open_rasterio(path, masked=True).isel(band=band - 1)
    da.name = "depth"
    if da.rio.crs is None or da.rio.crs.is_geographic:
        raise UnprojectedCRSError(
            f"{path} has no projected CRS ({da.rio.crs}); reproject to a projected CRS first."
        )
    return da

In [ ]:
#| hide
import rasterio
import tempfile
from pathlib import Path
from fastcore.test import test_fail, test_eq

def _write_test_raster(path, crs, transform, nodata=-9999.0):
    data = np.array([[1.0, 2.0], [nodata, 4.0]], dtype="float32")
    with rasterio.open(
        path, "w", driver="GTiff", height=2, width=2, count=1,
        dtype="float32", crs=crs, transform=transform, nodata=nodata,
    ) as dst:
        dst.write(data, 1)

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)

    # projected CRS -> loads fine, named "depth", nodata cell becomes NaN
    projected_path = tmp / "projected.tif"
    _write_test_raster(projected_path, "EPSG:32629", rasterio.transform.from_origin(0, 10, 5, 5))
    da = load_bathymetry(projected_path)
    test_eq(da.name, "depth")
    assert np.isnan(da.values[1, 0])

    # geographic CRS -> rejected
    geographic_path = tmp / "geographic.tif"
    _write_test_raster(geographic_path, "EPSG:4326", rasterio.transform.from_origin(0, 10, 0.01, 0.01))
    test_fail(lambda: load_bathymetry(geographic_path), contains="projected CRS")

    # no CRS at all -> also rejected (this was the bug: previously slipped through silently)
    no_crs_path = tmp / "no_crs.tif"
    _write_test_raster(no_crs_path, None, rasterio.transform.from_origin(0, 10, 5, 5))
    test_fail(lambda: load_bathymetry(no_crs_path), contains="projected CRS")

In [ ]:
#| hide
# smoke test against real survey data
real_tile = Path("/home/anokye/Documents/Data/BantryBay_5m_tiles/BantryBay_5m_tile_row1_col2.tif")
if real_tile.exists():
    da_real = load_bathymetry(real_tile)
    test_eq(da_real.name, "depth")
    test_eq(da_real.ndim, 2)
    assert da_real.rio.crs is not None and not da_real.rio.crs.is_geographic

### `kernel`

The neighbourhood shape shared by every focal statistic below -- a disc
(`kernel(radius)`) or an annulus (`kernel(inner_radius, outer_radius)`), dispatched
on argument count via `plum`. Built from a Euclidean distance test, not a square
window, so `radius=1` gives the centre cell plus its 4-connected neighbours (5
cells), not the full 3x3 block.

In [ ]:
#| export
@dispatch
def kernel(radius: int):
    """Boolean disc mask of the given radius (in cells): True where dy^2+dx^2 <= radius^2."""
    r = int(radius)
    
    x, y = np.ogrid[-r: r+1, -r:r+1]
    dist = x**2 + y**2
    return dist <= r**2

@dispatch
def kernel(inner_radius: int, outer_radius: int):
    """Boolean annulus mask (in cells): True in the ring strictly beyond inner_radius,
    up to and including outer_radius. Equivalent to kernel(outer) with kernel(inner)
    subtracted out; the array is sized to the outer radius."""
    i_r = int(inner_radius)
    o_r = int(outer_radius)

    x, y = np.ogrid[-o_r: o_r+1, -o_r:o_r+1]
    dist = x**2 + y**2
    return (dist > i_r**2) & (dist <= o_r**2)


In [ ]:
#| hide
# disc radius=1 -> centre + 4-connected neighbours only (5 cells), not the full 3x3
disc1 = kernel(1)
test_eq(disc1.shape, (3, 3))
test_eq(disc1.sum(), 5)

# annulus cell count matches outer-disc-count minus inner-disc-count, at two radius pairs
for inner, outer in [(2, 5), (0, 3)]:
    ring = kernel(inner, outer)
    test_eq(ring.shape, (2 * outer + 1, 2 * outer + 1))
    test_eq(ring.sum(), kernel(outer).sum() - kernel(inner).sum())

### `focal_mean`

NaN-aware windowed mean over a `kernel` shape, via FFT convolution
(`scipy.signal.fftconvolve`) rather than direct spatial convolution -- BPI's
broad-scale annulus radii get large, and FFT convolution's cost doesn't scale with
kernel area the way direct convolution's does. Every other focal statistic in this
module (`focal_std`, `local_zscore`) is built on top of this one.

In [ ]:
#| export
def focal_mean(data: xr.DataArray, radius: int | list) -> xr.DataArray:
    """NaN-aware focal mean over a disc (radius=int) or annulus (radius=[inner, outer]) kernel.

    Cells with no valid data under the kernel (all-NaN neighbourhood, or entirely
    off the raster edge) come back as NaN rather than 0 or inf."""
    k = kernel(radius) if isinstance(radius, int) else kernel(*radius)

    valid_mask = ~np.isnan(data.values)
    values_filled = np.where(valid_mask, data.values, 0)
    valid_count = valid_mask.astype(np.float64)

    sum_values = fftconvolve(values_filled, k, mode="same")
    count_valid = fftconvolve(valid_count, k, mode="same")

    with np.errstate(invalid="ignore", divide="ignore"):
        mean = sum_values / count_valid
    mean[count_valid < 0.5] = np.nan  # no valid cells anywhere in the window

    return data.copy(data=mean)

In [ ]:
#| hide
from fastcore.test import test_close

# constant-value raster -> focal mean equals the constant everywhere in the interior,
# for both a disc and an annulus radius
const_da = xr.DataArray(np.full((20, 20), 5.0), dims=("y", "x"))
test_close(focal_mean(const_da, 3).values[5:15, 5:15], 5.0)
test_close(focal_mean(const_da, [2, 5]).values[8:12, 8:12], 5.0)

# sparse raster (mostly NaN, one small block of real data) -> NaN far away,
# finite and correct right at the data
sparse = np.full((20, 20), np.nan)
sparse[9:11, 9:11] = 5.0
sparse_da = xr.DataArray(sparse, dims=("y", "x"))
fm_sparse = focal_mean(sparse_da, 3)
assert np.isnan(fm_sparse.values[0, 0])       # no valid neighbours under the kernel there
assert not np.isnan(fm_sparse.values[9, 9])
test_close(fm_sparse.values[9, 9], 5.0)       # only real values under the kernel are all 5.0

# randomly-scattered NaN (~20%, plus a few forced positions) -> every finite output cell's
# value is bounded by its neighbourhood's own min/max, i.e. it's a real local average,
# not something contaminated by the NaN-as-0 fill trick leaking through
rng = np.random.default_rng(0)
noisy = rng.uniform(0, 10, (30, 30))
for pos in [(0, 2), (2, 4), (4, 1)]:
    noisy[pos] = np.nan
noisy[rng.random((30, 30)) < 0.20] = np.nan
noisy_da = xr.DataArray(noisy, dims=("y", "x"))
fm_noisy = focal_mean(noisy_da, 2)
finite = ~np.isnan(fm_noisy.values)
assert finite.any() and (fm_noisy.values[finite] >= 0).all() and (fm_noisy.values[finite] <= 10).all()

In [ ]:
#| hide
# smoke test against real survey data
if real_tile.exists():
    da_real = load_bathymetry(real_tile)
    fm_real = focal_mean(da_real, 5)
    test_eq(fm_real.shape, da_real.shape)
    assert np.isfinite(fm_real.values).any()

### `focal_std`

Windowed standard deviation, built by calling `focal_mean` twice (once on the data,
once on the data squared) rather than a separate NaN-masking implementation --
`Var[X] = E[X^2] - E[X]^2`. See the docstring for the floating-point cancellation
fix this needed.

In [ ]:
#| export
def focal_std(data: xr.DataArray, radius: int | list) -> xr.DataArray:
    """NaN-aware focal (windowed) standard deviation, via Var[X] = E[X^2] - E[X]^2.

    Clips negative variance from floating-point cancellation (a near-zero true
    variance can round slightly negative) before the square root, so a flat or
    near-flat neighbourhood gives 0, not NaN."""
    mean = focal_mean(data, radius)
    mean_sq = focal_mean(data ** 2, radius)
    variance = (mean_sq - mean ** 2).clip(min=0)
    return np.sqrt(variance)

In [ ]:
#| hide
# constant-value raster -> zero local variance everywhere in the interior. This is
# also the case most likely to trip the E[X^2]-E[X]^2 cancellation bug if the clip
# were missing: true variance is exactly 0, so any negative floating-point noise
# would sqrt to NaN instead of 0.
test_close(focal_std(const_da, 3).values[5:15, 5:15], 0.0)

# ground truth: hand-slice the same disc window out of a random patch and compare
# against numpy's own std over exactly those cells
rng = np.random.default_rng(1)
patch = rng.uniform(0, 10, (15, 15))
patch_da = xr.DataArray(patch, dims=("y", "x"))
radius = 3
k = kernel(radius)
cy, cx = 7, 7  # interior cell, kernel fits fully inside the array here
window_vals = patch[cy - radius:cy + radius + 1, cx - radius:cx + radius + 1][k]
test_close(focal_std(patch_da, radius).values[cy, cx], window_vals.std())

### `global_zscore`

Whole-field standardisation -- one mean and one std computed over the entire
raster, not windowed. Needed before comparing statistics computed at different
radii or from different algorithms: TPI at radius 10 isn't on the same scale as TPI
at radius 50 without this.

In [ ]:
#| export
def global_zscore(data: xr.DataArray) -> xr.DataArray:
    """Whole-field NaN-aware z-score: (data - mean) / std, computed once over the
    entire array -- not windowed. See local_zscore for a windowed version."""
    with np.errstate(invalid="ignore"):
        global_mean = np.nanmean(data.values)
        global_std = np.nanstd(data.values)

    z = (data - global_mean) / global_std
    z.name = "global_zscore"
    return z

In [ ]:
#| hide
known_da = xr.DataArray(np.array([1.0, 2.0, 3.0, 4.0, 5.0]), dims=("x",))
gz = global_zscore(known_da)
test_close(float(gz.mean()), 0.0)
test_close(float(gz.std()), 1.0)

### `local_zscore`

Windowed standardisation, built from `focal_mean` and `focal_std`. This is the one
with the scale-dependent floating-point tolerance -- a fixed epsilon here silently
passed at one depth magnitude and failed at another before the fix (see the
docstring and the test cell below, which checks both magnitudes rather than one).

In [ ]:
#| export
def local_zscore(data: xr.DataArray, radius: int | list) -> xr.DataArray:
    """NaN-aware local (windowed) z-score: (data - focal_mean) / focal_std.

    A window with no real local variance (flat neighbourhood, or only one valid
    cell) gives NaN rather than inf or a bogus huge value -- there's no meaningful
    z-score without spread. "No real variance" is judged against a tolerance that
    scales with the local mean's magnitude, not a fixed number: focal_std's own
    E[X^2]-E[X]^2 computation leaves floating-point noise on the order of 1e-8x the
    data's magnitude even when the true variance is exactly 0 (checked empirically
    across depth magnitudes from ~5 to ~5000), so a fixed threshold that works at
    one depth scale silently fails at another."""
    mean = focal_mean(data, radius)
    std = focal_std(data, radius)

    with np.errstate(invalid="ignore", divide="ignore"):
        z = (data - mean) / std

    noise_floor = 1e-8 + 1e-6 * np.abs(mean)
    return z.where(std > noise_floor)

In [ ]:
#| hide
# zero local variance -> NaN, not inf (mirrors focal_mean's zero-count -> NaN test).
# Checked at two very different magnitudes, not just one: focal_std's floating-point
# noise floor scales with the data's own magnitude, so a fixed epsilon that happens
# to work at one depth scale can silently miss this at another.
for magnitude in (5.0, 5000.0):
    const_at_scale = xr.DataArray(np.full((20, 20), magnitude), dims=("y", "x"))
    lz = local_zscore(const_at_scale, 3)
    assert np.all(np.isnan(lz.values[5:15, 5:15])), f"failed at magnitude={magnitude}"

# real local variance -> finite, sensibly-scaled z-scores away from the raster edge
lz_patch = local_zscore(patch_da, radius)
interior = lz_patch.values[radius:-radius, radius:-radius]
assert np.isfinite(interior).all()
assert np.abs(interior).max() < 10  # loose sanity bound, not a statistical claim

# real-data smoke test, same tile as everything else
if real_tile.exists():
    fs_real = focal_std(da_real, 5)
    lz_real = local_zscore(da_real, 5)
    gz_real = global_zscore(da_real)
    for out in (fs_real, lz_real, gz_real):
        test_eq(out.shape, da_real.shape)
        assert np.isfinite(out.values).any()

### `polygonize_mask`

The raster-to-vector boundary every future detection algorithm (TPI/BPI, Openness,
Geomorphons, LMI/CI) will eventually go through -- turns a boolean mask into a
`GeoDataFrame` of polygons, with minimum-mapping-unit filtering and an explicit
`connectivity` choice, since disc/annulus kernel output can produce diagonal-only
touches between True cells that 4-connectivity (the default) would split into
separate features.

In [ ]:
#| export
def polygonize_mask(
    mask: np.ndarray, transform, crs, area_threshold_m2: float = 0.0, connectivity: int = 4
) -> gpd.GeoDataFrame:
    """Vectorise a boolean raster mask into a GeoDataFrame, dropping polygons under
    `area_threshold_m2`. `connectivity` (4 or 8) controls whether two True cells that
    only touch diagonally count as one connected feature or two -- disc/annulus
    kernel output can produce diagonal-only touches, so this is worth setting
    deliberately rather than trusting the default."""
    mask_u8 = mask.astype(np.uint8)
    geoms = [
        shapely_shape(geom)
        for geom, value in rio_shapes(mask_u8, mask=mask, transform=transform, connectivity=connectivity)
        if value == 1
    ]
    gdf = gpd.GeoDataFrame(geometry=geoms, crs=crs)
    gdf["area_m2"] = gdf.geometry.area
    gdf = gdf.loc[gdf["area_m2"] >= area_threshold_m2].reset_index(drop=True)
    gdf.insert(0, "featID", np.arange(1, len(gdf) + 1))
    return gdf

In [ ]:
#| hide
# known shape -> exactly one polygon, area matches cell_size^2 * true_cell_count exactly,
# CRS preserved, featID assigned
test_transform = rasterio.transform.from_origin(0, 50, 5, 5)  # 5 m cells
test_mask = np.zeros((10, 10), dtype=bool)
test_mask[2:5, 2:5] = True  # 3x3 block = 9 cells
gdf = polygonize_mask(test_mask, test_transform, "EPSG:32629")
test_eq(len(gdf), 1)
test_close(gdf.geometry.area.iloc[0], 9 * 5 * 5)
test_eq(str(gdf.crs), "EPSG:32629")
test_eq(list(gdf["featID"]), [1])

# MMU filtering: an isolated 1-cell speckle (25 m^2) is dropped at a 100 m^2
# threshold, a real 9-cell block (225 m^2) survives
speckled_mask = np.zeros((10, 10), dtype=bool)
speckled_mask[2:5, 2:5] = True
speckled_mask[8, 8] = True
gdf_filtered = polygonize_mask(speckled_mask, test_transform, "EPSG:32629", area_threshold_m2=100.0)
test_eq(len(gdf_filtered), 1)
test_close(gdf_filtered.geometry.area.iloc[0], 225.0)

# connectivity actually matters: two diagonally-touching single cells are one
# feature at connectivity=8, two separate features at connectivity=4 (the default)
diag_mask = np.zeros((10, 10), dtype=bool)
diag_mask[3, 3] = True
diag_mask[4, 4] = True
test_eq(len(polygonize_mask(diag_mask, test_transform, "EPSG:32629", connectivity=4)), 2)
test_eq(len(polygonize_mask(diag_mask, test_transform, "EPSG:32629", connectivity=8)), 1)

# early end-to-end smoke test: raw statistic -> standardise -> threshold -> polygonize,
# on real data, even before a dedicated TPI module exists
if real_tile.exists():
    tpi_like = da_real - focal_mean(da_real, 5)
    high_mask = (global_zscore(tpi_like).values >= 1.0)
    gdf_real = polygonize_mask(high_mask, da_real.rio.transform(), da_real.rio.crs)
    # compare CRS objects for actual equivalence, not string reprs -- geopandas
    # normalises crs to pyproj.CRS (str() gives full WKT), rasterio's own .rio.crs
    # is a different CRS class (str() gives "EPSG:32629"); same CRS, different
    # string formats, so only object equality is a meaningful check here
    assert gdf_real.crs == da_real.rio.crs
    assert (gdf_real.geometry.area > 0).all()

### `show_map`

Shared plotting helper for every module's demo/exploration cells -- one raster, one
or more vector layers, or both overlaid, via `hvplot`/`holoviews`. Empty
GeoDataFrames are skipped rather than plotted as a blank layer, since that's the
common case once MMU filtering or a tight threshold has legitimately produced zero
features -- but a call where *every* layer is empty (or nothing was passed at all)
raises rather than silently returning a blank plot.

In [ ]:
#| export
_DEFAULT_VECTOR_COLORS = [
    "#E08A52",  # Muted Orange (Original)
    "#6FC3E0",  # Soft Sky Blue (Original)
    "#8FD1B8",  # Mint Green (Original)
    "#B5A6E0",  # Lavender Purple (Original)
    "#E26D9B",  # Raspberry Pink
    "#F4D068",  # Warm Yellow
    "#76A035",  # Olive Green
    "#C874D9",  # Orchid Magenta
    "#4A7BB0",  # Steel Blue
    "#D95D5D",  # Terracotta Red
]


def show_map(
    raster: xr.DataArray | None = None,
    vectors: dict[str, gpd.GeoDataFrame] | None = None,
    *,
    title: str = "",
    raster_cmap: str = "viridis",
    raster_clabel: str = "",
    raster_clim: tuple[float, float] | None = None,
    vector_colors: list[str] | None = None,
    frame_width: int = 750,
    frame_height: int = 420,
):
    """Plot a raster alone, vector layer(s) alone, or both overlaid. `vectors` maps a
    legend label to a GeoDataFrame (e.g. {"Bathymetric High": highs, "Bathymetric Low":
    lows}); empty GeoDataFrames are skipped rather than plotted as an empty layer."""
    if raster is None and not vectors:
        raise InvalidParameterError("show_map needs at least `raster` or `vectors` to plot.")

    layers = []
    if raster is not None:
        raster_kwargs = dict(
            x="x", y="y", cmap=raster_cmap, aspect="equal", rasterize=True,
            clabel=raster_clabel, frame_width=frame_width, frame_height=frame_height,
        )
        if raster_clim is not None:
            raster_kwargs["clim"] = raster_clim
        layers.append(raster.hvplot.image(**raster_kwargs))

    if vectors:
        palette = vector_colors or _DEFAULT_VECTOR_COLORS
        for (label, gdf), color in zip(vectors.items(), itertools.cycle(palette)):
            if len(gdf) == 0:
                continue
            layers.append(
                gdf.hvplot(geo=False, color=None, line_color=color, line_width=1.2, fill_alpha=0, label=label)
            )

    if not layers:
        raise InvalidParameterError("show_map has nothing to plot -- all provided vectors were empty.")

    plot = layers[0]
    for layer in layers[1:]:
        plot = plot * layer

    opts = {}
    if title:
        opts["title"] = title
    if vectors:
        opts["legend_position"] = "right"
    if opts:
        plot = plot.opts(**opts)
    return plot

In [ ]:
#| hide
# nothing to plot at all -> InvalidParameterError, not a confusing downstream error
test_fail(lambda: show_map(), contains="needs at least")

# vectors given but every GeoDataFrame is empty -> InvalidParameterError, not a
# silently-blank plot
empty_gdf = gpd.GeoDataFrame(geometry=[], crs="EPSG:32629")
test_fail(lambda: show_map(vectors={"Empty": empty_gdf}), contains="nothing to plot")

# raster alone, vectors alone, and both together (with one empty layer correctly
# skipped rather than erroring) all return a plot without raising
assert show_map(raster=const_da) is not None
assert show_map(vectors={"Test": gdf}) is not None
assert show_map(raster=const_da, vectors={"Test": gdf, "Skipped (empty)": empty_gdf}, title="check") is not None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()